## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> Unit of Analysis: One row represents exactly one unique content page snapshot identified by content_id.

> Time Window: Mid-panel observation window (specifically month = '2026-03').

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import duckdb
import os, sys

# Setup repository path safely for Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load dataset after cloning repository
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
con = duckdb.connect()

# Verify Grain (Zero duplicate content_ids)
q_grain = "SELECT content_id, COUNT(*) as cnt FROM df GROUP BY content_id HAVING COUNT(*) > 1;"
duplicates = con.execute(q_grain).df()

# Basic slice metrics
total_rows = len(df)
declining_rows = len(df[df['trend_direction'] == 'down'])

print(f"Total Rows in Slice: {total_rows}")
print(f"Grain Check - Duplicates (Must be 0): {len(duplicates)}")
print(f"Pages in Decline: {declining_rows} ({(declining_rows/total_rows)*100:.1f}%)")

Total Rows in Slice: 30000
Grain Check - Duplicates (Must be 0): 0
Pages in Decline: 16262 (54.2%)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features:** `word_count`, `content_age_days`, `impressions_90d`, `avg_position`, `ctr`
* **Label / Proxy:** `is_declining` (derived from `trend_direction == 'down'`)
* **Context:** `content_id`, `client_id`
* **Excluded:** `trend_pct` — *Excluded because it measures the direct percentage change of performance, causing severe feature leakage if fed into the model.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Select and inspect sample data from each of our 4 buckets
sample_fields = [
    'content_id', 'client_id',         # Context
    'word_count', 'content_age_days',  # Features
    'impressions_90d', 'avg_position', # Features
    'ctr',                             # Feature
    'trend_direction',                 # Label source
    'trend_pct'                        # Excluded field
]

print("Schema Inspection of Categorized Buckets:")
df[sample_fields].head()

Schema Inspection of Categorized Buckets:


,content_id,client_id,word_count,content_age_days,impressions_90d,avg_position,ctr,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3221.0,187,3803,10.6,0.76,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,2481.0,445,15320,20.3,0.05,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,3515.0,141,12581,36.5,0.09,down,-60.9
3,content_331d6c4de07b,client_19581e27de,NaN,463,11751,6.2,0.49,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,2803.0,263,19140,44.0,0.13,down,-34.7


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

* **Grain Verification:** Grouped by `content_id` with `HAVING COUNT(*) > 1` using DuckDB; returned 0 rows, confirming strict uniqueness.
* **Slice Counts & Availability:** Evaluated row survival using `IS TRUE` filter logic across essential features.
* **Missingness Patterns:** Measured null percentage across dataset fields to ensure baseline data quality.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()

# Query 1: Verify Grain Uniqueness (Duplicate count must be 0)
q_grain = """
SELECT content_id, COUNT(*) as cnt
FROM df
GROUP BY content_id
HAVING COUNT(*) > 1;
"""
grain_violations = len(con.execute(q_grain).df())
print(f"1. Grain Violation Count (Duplicates): {grain_violations}")

# Query 2: Slice Row Count & Availability Check (IS TRUE survival filter)
q_availability = """
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN (word_count IS NOT NULL AND impressions_90d IS NOT NULL) IS TRUE THEN 1 END) as survived_rows,
    ROUND(COUNT(CASE WHEN (word_count IS NOT NULL AND impressions_90d IS NOT NULL) IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as survival_pct
FROM df;
"""
print("\n2. Slice Row Count & Availability (IS TRUE Survival Filter):")
print(con.execute(q_availability).df())

# Query 3: Missingness Patterns
q_missing = """
SELECT
    AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0.0 END) as missing_word_count_pct,
    AVG(CASE WHEN ctr IS NULL THEN 1.0 ELSE 0.0 END) as missing_ctr_pct
FROM df;
"""
print("\n3. Missingness Percentage Check:")
print(con.execute(q_missing).df())

1. Grain Violation Count (Duplicates): 0

2. Slice Row Count & Availability (IS TRUE Survival Filter):
   total_rows  survived_rows  survival_pct
0       30000          22301         74.34

3. Missingness Percentage Check:
   missing_word_count_pct  missing_ctr_pct
0                0.256633              0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced History:** New content pages (`content_age_days < 90`) lack a complete 90-day historical window, creating structural variance compared to mature pages.
* **Window Overlaps:** Metrics spanning 30-day and 90-day windows overlap in time, meaning they represent correlated moving averages rather than independent observations.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()

# Query to verify data limits: Page age distribution (New vs Mature pages)
q_limits = """
SELECT
    COUNT(CASE WHEN content_age_days < 90 THEN 1 END) as new_pages_under_90d,
    COUNT(CASE WHEN content_age_days >= 90 THEN 1 END) as mature_pages_90d_plus,
    ROUND(COUNT(CASE WHEN content_age_days < 90 THEN 1 END) * 100.0 / COUNT(*), 2) as pct_new_pages
FROM df;
"""

print("Data Limit Verification — New vs Mature Page Distribution:")
print(con.execute(q_limits).df())

Data Limit Verification — New vs Mature Page Distribution:
   new_pages_under_90d  mature_pages_90d_plus  pct_new_pages
0                    0                  30000            0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.